In [3]:
import torch 
import torch.nn as nn
from model import TransformerBlock, LayerNorm, GELU, FeedForward, MultiHeadAttention

In [4]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12, 
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [5]:
class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding = nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.position_embedding = nn.Embedding(config["context_length"], config["emb_dim"])
        self.dropout = nn.Dropout(config["drop_rate"])
        self.transformer_block = nn.Sequential(
            *[TransformerBlock(config) for _ in range(config["n_layers"])]
        )
        self.final_norm = LayerNorm(config["emb_dim"])
        self.output_head = nn.Linear(
            config["emb_dim"], config["vocab_size"], bias=False
        )

    def forward(self, x):
        batch_size, seq_length = x.shape
        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(torch.arange(seq_length, device=x.device))
        x = token_emb + pos_emb
        x = self.dropout(x)
        x = self.transformer_block(x)
        x = self.final_norm(x)
        logits = self.output_head(x)
        return logits

In [6]:
torch.manual_seed(123)
gpt_model = GPTModel(GPT_CONFIG_124M)
sample_input = torch.randint(0, GPT_CONFIG_124M["vocab_size"], (2, 10))
output_logits = gpt_model(sample_input)
print("Output logits shape:", output_logits.shape)
print("Input shape:", sample_input.shape)

Output logits shape: torch.Size([2, 10, 50257])
Input shape: torch.Size([2, 10])


In [7]:
total_params = sum(p.numel() for p in gpt_model.parameters())
print(f"Total parameters in GPT-2 124M model: {total_params:,}")

Total parameters in GPT-2 124M model: 155,922,432


# Generating Text from Output Tokens

In [8]:
def generate_text_simple(model, idx, max_new_tokens, context_size):

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probs, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [9]:
import tiktoken
start_context = "Hello, I am"
tokenizer = tiktoken.get_encoding("gpt2")
input_ids = tokenizer.encode(start_context)
print("Input token IDs:", input_ids)
encoded_input = torch.tensor(input_ids).unsqueeze(0)
print("Encodede input shape:", encoded_input.shape)


Input token IDs: [15496, 11, 314, 716]
Encodede input shape: torch.Size([1, 4])


In [10]:
gpt_model.eval()
out = generate_text_simple(
    model=gpt_model,
    idx=encoded_input,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)
print("Output : ", out)
print("Output Length: ", len(out[0]))

Output :  tensor([[15496,    11,   314,   716, 16124, 50051, 40723, 48036, 43442, 19892]])
Output Length:  10


In [11]:
decoded_output = tokenizer.decode(out[0].tolist())
print("Decoded Output:", decoded_output)

Decoded Output: Hello, I am deriv Sins Activision locality Laurelolder


In [12]:
### 
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12, 
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [13]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (token_embedding): Embedding(50257, 768)
  (position_embedding): Embedding(256, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer_block): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layer): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
       

In [14]:
import tiktoken

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model = model,
    idx = text_to_token_ids(start_context, tokenizer),
    max_new_tokens = 10,
    context_size = GPT_CONFIG_124M["context_length"]
)

print("Output Text: \n ", token_ids_to_text(token_ids, tokenizer))

Output Text: 
  Every effort moves you Claytonanonzee barrelsJustinibly Blazers convertfulnessirling


# Implementing Loss functions

In [15]:
inputs = torch.tensor([[16833, 3626, 6100],
                       [40, 1107, 588]])

targets = torch.tensor([[3626, 6100, 345],
                        [1107, 588, 11311]])

In [16]:
with torch.no_grad():
    logits = model(inputs)
    print("Logits shape:", logits.shape)

Logits shape: torch.Size([2, 3, 50257])


In [18]:
token_ids = torch.argmax(logits, dim=-1, keepdim=True)
print("Predicted token IDs:\n", token_ids)

Predicted token IDs:
 tensor([[[19663],
         [25000],
         [22748]],

        [[33559],
         [ 4054],
         [ 1923]]])


In [21]:
print("Target batch 1 :", token_ids_to_text(targets[0], tokenizer))
print("Predicted batch 1 :", token_ids_to_text(token_ids[0].flatten(), tokenizer))

Target batch 1 :  effort moves you
Predicted batch 1 :  Mechanical MONowntown


In [26]:
print("logits shape:", logits.shape)
print("targets shape:", targets.shape)

logits shape: torch.Size([2, 3, 50257])
targets shape: torch.Size([2, 3])


In [27]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Flattened logits shape:", logits_flat.shape)
print("Flattened targets shape:", targets_flat.shape)

Flattened logits shape: torch.Size([6, 50257])
Flattened targets shape: torch.Size([6])


In [28]:
loss = nn.CrossEntropyLoss()(logits_flat, targets_flat)
print("Loss:", loss.item())

Loss: 10.48353099822998
